In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/complaints.csv")

print(df.shape)
df.head()

C:\Users\hp-pc\AppData\Local\Temp\ipykernel_17284\2675233770.py:3: DtypeWarning: Columns (0: Consumer disputed?) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/complaints.csv")


Check important columns:

In [3]:
df.columns

Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Consumer consent provided?',
       'Submitted via', 'Date sent to company', 'Company response to consumer',
       'Timely response?', 'Consumer disputed?', 'Complaint ID'],
      dtype='str')

You should locate columns similar to

Step 4: Basic Dataset Inspection

Check missing values:

In [5]:
df.isnull().sum()

Date received                         0
Product                               0
Sub-product                      235295
Issue                                 6
Sub-issue                        839522
Consumer complaint narrative    6629041
Company public response         4770207
Company                               0
State                             54516
ZIP code                          30228
Tags                            8981029
Consumer consent provided?      1649561
Submitted via                         0
Date sent to company                  0
Company response to consumer         20
Timely response?                      0
Consumer disputed?              8841498
Complaint ID                          0
dtype: int64

Check data types:

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9609797 entries, 0 to 9609796
Data columns (total 18 columns):
 #   Column                        Dtype
---  ------                        -----
 0   Date received                 str  
 1   Product                       str  
 2   Sub-product                   str  
 3   Issue                         str  
 4   Sub-issue                     str  
 5   Consumer complaint narrative  str  
 6   Company public response       str  
 7   Company                       str  
 8   State                         str  
 9   ZIP code                      str  
 10  Tags                          str  
 11  Consumer consent provided?    str  
 12  Submitted via                 str  
 13  Date sent to company          str  
 14  Company response to consumer  str  
 15  Timely response?              str  
 16  Consumer disputed?            str  
 17  Complaint ID                  int64
dtypes: int64(1), str(17)
memory usage: 1.3 GB


Check duplicates:

In [7]:
df.duplicated().sum()

np.int64(0)

Step 5: Analyze Product Distribution

In [1]:
import matplotlib.pyplot as plt

product_counts = df['Product'].value_counts()

plt.figure(figsize=(10,6))
product_counts.plot(kind='bar')
plt.title("Complaint Distribution by Product")
plt.show()

NameError: name 'df' is not defined

Step 6: Count Narratives vs Missing Narratives

In [9]:
narrative_col = "Consumer complaint narrative"

with_narrative = df[narrative_col].notna().sum()
without_narrative = df[narrative_col].isna().sum()

print(with_narrative)
print(without_narrative)

2980756
6629041


Visualization

In [1]:
plt.pie(
    [with_narrative, without_narrative],
    labels=["Has Narrative","No Narrative"],
    autopct="%1.1f%%"
)
plt.show()

NameError: name 'plt' is not defined

Step 7: Analyze Narrative Length

Create word count feature:

In [ ]:
df["word_count"] = (
    df[narrative_col]
    .fillna("")
    .apply(lambda x: len(x.split()))
)

Statistics

In [ ]:
df["word_count"].describe()

Histogram

In [ ]:
plt.figure(figsize=(10,6))
plt.hist(df["word_count"], bins=50)
plt.title("Narrative Length Distribution")
plt.xlabel("Word Count")
plt.ylabel("Frequency")
plt.show()

Step 8: Filter Required Products

In [ ]:
df["Product"].unique()

Then filter:

In [ ]:
selected_products = [
    "Credit card",
    "Personal loan",
    "Savings account",
    "Money transfer"
]

filtered_df = df[
    df["Product"].isin(selected_products)
]

Check size:

In [ ]:
filtered_df.shape

Step 9: Remove Empty Narratives

In [ ]:
filtered_df = filtered_df[
    filtered_df[narrative_col].notna()
]

Also remove blanks:

In [ ]:
filtered_df = filtered_df[
    filtered_df[narrative_col].str.strip() != ""
]

Step 10: Create Cleaning Function

In [ ]:
import re

def clean_text(text):

    text = text.lower()

    text = re.sub(
        r"i am writing to file a complaint.*?",
        "",
        text
    )

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

Apply

In [ ]:
filtered_df["cleaned_narrative"] = (
    filtered_df[narrative_col]
    .apply(clean_text)
)

Step 11: Save Cleaned Dataset

In [ ]:
filtered_df.to_csv(
    "data/filtered_complaints.csv",
    index=False
)

Task 2: Chunking, Embeddings & Vector Store
Step 2: Load Cleaned Data

In [ ]:
import pandas as pd

df = pd.read_csv(
    "data/filtered_complaints.csv"
)

df.shape

Step 3: Create Stratified Sample

In [ ]:
from sklearn.model_selection import train_test_split

Sampling

In [ ]:
sample_size = 12000

sample_df = (
    df.groupby("Product", group_keys=False)
    .apply(
        lambda x:
        x.sample(
            int(
                sample_size *
                len(x)/len(df)
            ),
            random_state=42
        )
    )
)

Verify

In [ ]:
sample_df["Product"].value_counts(normalize=True)

Step 4: Text Chunking

In [ ]:
chunk_size = 500
chunk_overlap = 50

Create Splitter

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

Step 5: Generate Chunks

In [ ]:
documents = []

for _, row in sample_df.iterrows():

    chunks = splitter.split_text(
        row["cleaned_narrative"]
    )

    for i, chunk in enumerate(chunks):

        documents.append({
            "complaint_id": row["Complaint ID"],
            "product": row["Product"],
            "chunk": chunk,
            "chunk_index": i
        })

Convert to DataFrame:

In [ ]:
chunks_df = pd.DataFrame(documents)

chunks_df.head()

Step 6: Load Embedding Model

Challenge recommendation:

In [ ]:
all-MiniLM-L6-v2

Load

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Step 7: Generate Embeddings

In [ ]:
texts = chunks_df["chunk"].tolist()

embeddings = model.encode(
    texts,
    show_progress_bar=True
)

Shape

In [ ]:
embeddings.shape

Expected

In [ ]:
(number_of_chunks, 384)

Step 8A: Create FAISS Index

In [ ]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(embeddings).astype("float32")
)

Save

In [ ]:
faiss.write_index(
    index,
    "vector_store/faiss_index.bin"
)

Step 8B (Recommended): Create ChromaDB

In [ ]:
import chromadb

client = chromadb.PersistentClient(
    path="vector_store/chroma_db"
)

collection = client.get_or_create_collection(
    "complaints"
)

Add records:

In [ ]:
for i, row in chunks_df.iterrows():

    collection.add(
        ids=[str(i)],
        documents=[row["chunk"]],
        metadatas=[{
            "complaint_id":
            str(row["complaint_id"]),

            "product":
            row["product"]
        }]
    )

Step 9: Test Retrieval

In [ ]:
query = "credit card billing issues"

results = collection.query(
    query_texts=[query],
    n_results=5
)

print(results["documents"])